# Landslide combined class step 05: expected annual damage calculations (maximum scenario)

Calculates expected annual damages from the combined-class maximum-scenario direct-damage outputs produced by step 02.


In [ ]:
import re
import sys
from pathlib import Path

from IPython.display import display
import numpy as np
import pandas as pd

base_path = Path('/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers')
robyn_library_path = base_path / 'robyns_libraries'
if str(robyn_library_path) not in sys.path:
    sys.path.append(str(robyn_library_path))

import Robyn_river_floods

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

# Reporting in USD only
JMD_PER_USD = 150.0
USD_PER_JMD = 1.0 / JMD_PER_USD


In [ ]:
# Core paths
results_path = base_path / 'dphil_paper_3/results/02_damage_estimates/landslide_damages/results_landslide_maximum_scenario_combined_class'
direct_damages_path = results_path / 'direct_damages'
damage_estimates_path = results_path / 'damage_estimates'
network_metadata_file = base_path / 'dphil_common_cross_cutting/common_incoming_data/networks/network_layers_hazard_intersections_details.csv'

if not direct_damages_path.exists():
    raise FileNotFoundError(f'Missing folder: {direct_damages_path}')
if not network_metadata_file.exists():
    raise FileNotFoundError(f'Missing file: {network_metadata_file}')

print(f'direct_damages_path: {direct_damages_path}')
print(f'network_metadata_file: {network_metadata_file}')


In [ ]:
# Discover all direct-damage parquet files produced for sensitivity parameter set 0
parquet_files = sorted(direct_damages_path.rglob('*_direct_damages_parameter_set_0_combined_class.parquet'))

if not parquet_files:
    raise FileNotFoundError(f'No direct damage parquet files found under {direct_damages_path}')

print(f'Found {len(parquet_files)} direct-damage files.')
pd.DataFrame({'parquet_file': [str(parquet_file) for parquet_file in parquet_files]})


In [ ]:
# Read network metadata and build expected file mapping (asset/layer/id column)
network_details = pd.read_csv(network_metadata_file)
required_columns = ['asset_gpkg', 'asset_layer', 'asset_description', 'asset_id_column', 'sector']
missing_columns = [column_name for column_name in required_columns if column_name not in network_details.columns]
if missing_columns:
    raise KeyError(f'Missing required columns in network metadata: {missing_columns}')

network_details = network_details[required_columns].drop_duplicates().copy()
network_details['folder_name'] = network_details['asset_gpkg'] + '_' + network_details['asset_layer']
network_details['expected_parquet'] = network_details['folder_name'].apply(
    lambda folder: direct_damages_path / folder / f'{folder}_direct_damages_parameter_set_0_combined_class.parquet'
)
network_details['exists'] = network_details['expected_parquet'].apply(lambda parquet_file: parquet_file.exists())

display(network_details[['sector', 'asset_description', 'asset_gpkg', 'asset_layer', 'asset_id_column', 'exists']].sort_values(['sector', 'asset_gpkg', 'asset_layer']))

missing_files = network_details.loc[~network_details['exists'], ['asset_gpkg', 'asset_layer', 'expected_parquet']]
if len(missing_files) > 0:
    print('Missing expected files:')
    display(missing_files)
else:
    print('All expected direct-damage files are present.')


In [ ]:
# Load each damage table and verify combined-class landslide return-period columns
hazard_pattern = re.compile(r'^landslide_combined_class_(baseline|deforestation|reafforestation)_rp_(\d+)$')
required_scenarios = ['baseline', 'deforestation', 'reafforestation']
required_return_periods = [5, 10, 25, 50, 100]

loaded_damage_tables = {}
load_summary_rows = []

for row in network_details.itertuples(index=False):
    parquet_path = row.expected_parquet
    if not parquet_path.exists():
        continue

    damage_data = pd.read_parquet(parquet_path)
    landslide_columns = [column_name for column_name in damage_data.columns if hazard_pattern.match(column_name)]

    scenario_return_periods = {scenario_name: [] for scenario_name in required_scenarios}
    for column_name in landslide_columns:
        match = hazard_pattern.match(column_name)
        scenario_name = match.group(1)
        scenario_return_periods[scenario_name].append(int(match.group(2)))

    scenario_return_periods = {
        scenario_name: sorted(set(return_periods))
        for scenario_name, return_periods in scenario_return_periods.items()
    }
    scenario_return_period_counts = {
        scenario_name: len(return_periods)
        for scenario_name, return_periods in scenario_return_periods.items()
    }

    key = f'{row.asset_gpkg}_{row.asset_layer}'
    loaded_damage_tables[key] = damage_data

    missing_scenarios = [
        scenario_name
        for scenario_name, return_period_count in scenario_return_period_counts.items()
        if return_period_count == 0
    ]
    missing_return_periods = {
        scenario_name: [return_period for return_period in required_return_periods if return_period not in scenario_return_periods[scenario_name]]
        for scenario_name in required_scenarios
    }
    unexpected_return_periods = {
        scenario_name: [return_period for return_period in scenario_return_periods[scenario_name] if return_period not in required_return_periods]
        for scenario_name in required_scenarios
    }
    return_period_issue_parts = []
    for scenario_name in required_scenarios:
        if missing_return_periods[scenario_name]:
            return_period_issue_parts.append(f'{scenario_name} missing {missing_return_periods[scenario_name]}')
        if unexpected_return_periods[scenario_name]:
            return_period_issue_parts.append(f'{scenario_name} unexpected {unexpected_return_periods[scenario_name]}')

    load_summary_rows.append({
        'table_key': key,
        'sector': row.sector,
        'subsector': row.asset_description,
        'rows': len(damage_data),
        'columns': len(damage_data.columns),
        'landslide_cols': len(landslide_columns),
        'baseline_rps': ', '.join(map(str, scenario_return_periods['baseline'])),
        'deforestation_rps': ', '.join(map(str, scenario_return_periods['deforestation'])),
        'reafforestation_rps': ', '.join(map(str, scenario_return_periods['reafforestation'])),
        'baseline_rp_cols': scenario_return_period_counts['baseline'],
        'deforestation_rp_cols': scenario_return_period_counts['deforestation'],
        'reafforestation_rp_cols': scenario_return_period_counts['reafforestation'],
        'missing_scenarios': ', '.join(missing_scenarios) if missing_scenarios else '',
        'return_period_issues': '; '.join(return_period_issue_parts),
        'parquet_path': str(parquet_path),
    })

load_summary = pd.DataFrame(load_summary_rows).sort_values(['sector', 'table_key']).reset_index(drop=True)
display(load_summary)

bad_tables = load_summary[(load_summary['missing_scenarios'] != '') | (load_summary['return_period_issues'] != '')]
if len(bad_tables) > 0:
    print('Some tables are missing expected landslide scenarios or return periods:')
    display(bad_tables[['table_key', 'missing_scenarios', 'return_period_issues']])
else:
    print('All loaded tables include baseline, deforestation, and reafforestation combined-class columns for expected return periods:', required_return_periods)


In [ ]:
# Optional preview: pick one loaded table
table_to_preview = 'roads_edges'  # change as needed

if table_to_preview not in loaded_damage_tables:
    print(f"'{table_to_preview}' not found. Available keys:")
    print(sorted(loaded_damage_tables.keys()))
else:
    preview_columns = [
        column_name for column_name in loaded_damage_tables[table_to_preview].columns
        if column_name.startswith('landslide_') and '_rp_' in column_name
    ]
    display(loaded_damage_tables[table_to_preview][preview_columns].head(10))


In [ ]:
# Confirm the EAD helper imported in the first executable cell
print(f'Using Robyn_river_floods from: {Robyn_river_floods.__file__}')


In [ ]:
# Compute asset-level EADs using all available landslide RPs for each scenario (USD)
asset_ead_rows = []

for row in network_details.itertuples(index=False):
    table_key = f'{row.asset_gpkg}_{row.asset_layer}'
    if table_key not in loaded_damage_tables:
        continue

    damage_df = loaded_damage_tables[table_key].copy()
    asset_id_col = row.asset_id_column

    if asset_id_col not in damage_df.columns:
        print(f"Skipping {table_key}: missing asset ID column '{asset_id_col}'")
        continue

    scenario_cols = {}
    scenario_rps = {}
    for scenario_name in ['baseline', 'deforestation', 'reafforestation']:
        cols = [
            column_name for column_name in damage_df.columns
            if column_name.startswith(f'landslide_combined_class_{scenario_name}_rp_')
        ]
        rps = []
        for column_name in cols:
            hazard_match = re.match(r'^landslide_combined_class_' + scenario_name + r'_rp_(\d+)$', column_name)
            if hazard_match:
                rps.append(int(hazard_match.group(1)))
        order = np.argsort(rps)
        scenario_cols[scenario_name] = [cols[sort_index] for sort_index in order] if len(cols) > 0 else []
        scenario_rps[scenario_name] = [rps[sort_index] for sort_index in order] if len(rps) > 0 else []

    if any(len(scenario_cols[scenario_name]) == 0 for scenario_name in required_scenarios):
        print(f"Skipping {table_key}: missing one or more scenario RP columns")
        continue

    needed_cols = [asset_id_col] + scenario_cols['baseline'] + scenario_cols['deforestation'] + scenario_cols['reafforestation']
    grouped = damage_df[needed_cols].groupby(asset_id_col, as_index=False).sum().copy()

    for scenario_name, out_col in [
        ('baseline', 'EAD_Baseline_USD'),
        ('deforestation', 'EAD_Deforestation_USD'),
        ('reafforestation', 'EAD_Reafforestation_USD'),
    ]:
        rp_map = {
            col: f"rp{float(rp):.1f}"
            for col, rp in zip(scenario_cols[scenario_name], scenario_rps[scenario_name])
        }
        ead_input = grouped[scenario_cols[scenario_name]].rename(columns=rp_map)
        ead_jmd = Robyn_river_floods.calculate_ead(ead_input)
        grouped[out_col] = ead_jmd * USD_PER_JMD

    grouped['Deforestation_Change_EAD_USD'] = grouped['EAD_Deforestation_USD'] - grouped['EAD_Baseline_USD']
    grouped['Reafforestation_Change_EAD_USD'] = grouped['EAD_Reafforestation_USD'] - grouped['EAD_Baseline_USD']
    grouped['Reafforestation_Benefit_vs_Deforestation_USD'] = grouped['EAD_Deforestation_USD'] - grouped['EAD_Reafforestation_USD']

    grouped['Deforestation_Change_Share_of_Baseline'] = grouped['Deforestation_Change_EAD_USD'] / grouped['EAD_Baseline_USD']
    grouped['Reafforestation_Change_Share_of_Baseline'] = grouped['Reafforestation_Change_EAD_USD'] / grouped['EAD_Baseline_USD']
    grouped['Reafforestation_Benefit_Share_of_Deforestation'] = grouped['Reafforestation_Benefit_vs_Deforestation_USD'] / grouped['EAD_Deforestation_USD']

    grouped.loc[grouped['EAD_Baseline_USD'] <= 0, ['Deforestation_Change_Share_of_Baseline', 'Reafforestation_Change_Share_of_Baseline']] = pd.NA
    grouped.loc[grouped['EAD_Deforestation_USD'] <= 0, 'Reafforestation_Benefit_Share_of_Deforestation'] = pd.NA

    grouped['Sector'] = row.sector
    grouped['Subsector'] = row.asset_description
    grouped['Asset'] = row.asset_gpkg
    grouped['Layer'] = row.asset_layer
    grouped = grouped.rename(columns={asset_id_col: 'Asset_ID'})

    out_cols = [
        'Sector', 'Subsector', 'Asset', 'Layer', 'Asset_ID',
        'EAD_Baseline_USD', 'EAD_Deforestation_USD', 'EAD_Reafforestation_USD',
        'Deforestation_Change_EAD_USD', 'Reafforestation_Change_EAD_USD', 'Reafforestation_Benefit_vs_Deforestation_USD',
        'Deforestation_Change_Share_of_Baseline', 'Reafforestation_Change_Share_of_Baseline', 'Reafforestation_Benefit_Share_of_Deforestation'
    ]
    asset_ead_rows.append(grouped[out_cols])

asset_ead = pd.concat(asset_ead_rows, ignore_index=True) if asset_ead_rows else pd.DataFrame()

print(f'Asset rows with EAD (USD): {len(asset_ead):,}')
display(asset_ead.head(20))


In [ ]:
# Summaries: subsector and sector EAD (USD)
if asset_ead.empty:
    raise ValueError('asset_ead is empty; check missing columns or source files.')

sum_cols = [
    'EAD_Baseline_USD', 'EAD_Deforestation_USD', 'EAD_Reafforestation_USD',
    'Deforestation_Change_EAD_USD', 'Reafforestation_Change_EAD_USD', 'Reafforestation_Benefit_vs_Deforestation_USD'
]

subsector_ead_summary = (
    asset_ead
    .groupby(['Sector', 'Subsector'], as_index=False)[sum_cols]
    .sum()
)

sector_ead_summary = (
    asset_ead
    .groupby(['Sector'], as_index=False)[sum_cols]
    .sum()
)

for summary_df in [subsector_ead_summary, sector_ead_summary]:
    summary_df['Deforestation_Change_Share_of_Baseline'] = summary_df['Deforestation_Change_EAD_USD'] / summary_df['EAD_Baseline_USD']
    summary_df['Reafforestation_Change_Share_of_Baseline'] = summary_df['Reafforestation_Change_EAD_USD'] / summary_df['EAD_Baseline_USD']
    summary_df['Reafforestation_Benefit_Share_of_Deforestation'] = summary_df['Reafforestation_Benefit_vs_Deforestation_USD'] / summary_df['EAD_Deforestation_USD']

    summary_df.loc[summary_df['EAD_Baseline_USD'] <= 0, ['Deforestation_Change_Share_of_Baseline', 'Reafforestation_Change_Share_of_Baseline']] = pd.NA
    summary_df.loc[summary_df['EAD_Deforestation_USD'] <= 0, 'Reafforestation_Benefit_Share_of_Deforestation'] = pd.NA

print('Subsector EAD summary (USD):')
display(subsector_ead_summary.sort_values(['Sector', 'Subsector']))

print('Sector EAD summary (USD):')
display(sector_ead_summary.sort_values(['Sector']))


In [ ]:
# Save outputs (USD only)
damage_estimates_path.mkdir(parents=True, exist_ok=True)

asset_ead_output_file = damage_estimates_path / 'landslide_ead_asset_level_usd_combined_class.csv'
subsector_ead_output_file = damage_estimates_path / 'landslide_ead_subsector_summary_usd_combined_class.csv'
sector_ead_output_file = damage_estimates_path / 'landslide_ead_sector_summary_usd_combined_class.csv'

asset_ead.to_csv(asset_ead_output_file, index=False)
subsector_ead_summary.to_csv(subsector_ead_output_file, index=False)
sector_ead_summary.to_csv(sector_ead_output_file, index=False)

print(f'Saved: {asset_ead_output_file}')
print(f'Saved: {subsector_ead_output_file}')
print(f'Saved: {sector_ead_output_file}')


In [ ]:
# USD output preview
print('Asset-level EAD in USD:')
display(asset_ead.head(20))

print('Subsector EAD summary in USD:')
display(subsector_ead_summary.sort_values(['Sector', 'Subsector']))

print('Sector EAD summary in USD:')
display(sector_ead_summary.sort_values(['Sector']))


In [ ]:
# Percentage reporting tables for quick interpretation
subsector_pct = subsector_ead_summary.copy()
sector_pct = sector_ead_summary.copy()

for df in [subsector_pct, sector_pct]:
    df['Percent_Deforestation_Change_vs_Baseline'] = pd.NA
    df['Percent_Reafforestation_Change_vs_Baseline'] = pd.NA
    df['Percent_Reafforestation_Benefit_vs_Deforestation'] = pd.NA

    valid_baseline = df['EAD_Baseline_USD'] > 0
    df.loc[valid_baseline, 'Percent_Deforestation_Change_vs_Baseline'] = (
        100.0 * df.loc[valid_baseline, 'Deforestation_Change_EAD_USD'] / df.loc[valid_baseline, 'EAD_Baseline_USD']
    )
    df.loc[valid_baseline, 'Percent_Reafforestation_Change_vs_Baseline'] = (
        100.0 * df.loc[valid_baseline, 'Reafforestation_Change_EAD_USD'] / df.loc[valid_baseline, 'EAD_Baseline_USD']
    )

    valid_deforestation = df['EAD_Deforestation_USD'] > 0
    df.loc[valid_deforestation, 'Percent_Reafforestation_Benefit_vs_Deforestation'] = (
        100.0 * df.loc[valid_deforestation, 'Reafforestation_Benefit_vs_Deforestation_USD'] / df.loc[valid_deforestation, 'EAD_Deforestation_USD']
    )

print('Sector-level percentage summaries:')
display(
    sector_pct[[
        'Sector', 'EAD_Baseline_USD', 'EAD_Deforestation_USD', 'EAD_Reafforestation_USD',
        'Percent_Deforestation_Change_vs_Baseline', 'Percent_Reafforestation_Change_vs_Baseline',
        'Percent_Reafforestation_Benefit_vs_Deforestation'
    ]].sort_values('Sector').round({
        'Percent_Deforestation_Change_vs_Baseline': 2,
        'Percent_Reafforestation_Change_vs_Baseline': 2,
        'Percent_Reafforestation_Benefit_vs_Deforestation': 2,
    })
)

print('Subsector-level percentage summaries:')
display(
    subsector_pct[[
        'Sector', 'Subsector', 'EAD_Baseline_USD', 'EAD_Deforestation_USD', 'EAD_Reafforestation_USD',
        'Percent_Deforestation_Change_vs_Baseline', 'Percent_Reafforestation_Change_vs_Baseline',
        'Percent_Reafforestation_Benefit_vs_Deforestation'
    ]].sort_values(['Sector', 'Subsector']).round({
        'Percent_Deforestation_Change_vs_Baseline': 2,
        'Percent_Reafforestation_Change_vs_Baseline': 2,
        'Percent_Reafforestation_Benefit_vs_Deforestation': 2,
    })
)

sector_percentage_output_file = damage_estimates_path / 'landslide_ead_sector_summary_usd_with_pct_change_combined_class.csv'
subsector_percentage_output_file = damage_estimates_path / 'landslide_ead_subsector_summary_usd_with_pct_change_combined_class.csv'

sector_pct.to_csv(sector_percentage_output_file, index=False)
subsector_pct.to_csv(subsector_percentage_output_file, index=False)

print(f'Saved: {sector_percentage_output_file}')
print(f'Saved: {subsector_percentage_output_file}')


## Overall Totals (All Sectors)

Total EAD across all sectors for each scenario, plus total changes and percentages in readable USD units.

In [ ]:
# Overall totals across all sectors (USD + readable labels)

def format_usd_readable(value):
    if pd.isna(value):
        return 'NA'
    abs_value = abs(float(value))
    sign = '-' if float(value) < 0 else ''

    if abs_value >= 1_000_000_000:
        return f"{sign}${abs_value / 1_000_000_000:,.2f} billion"
    if abs_value >= 1_000_000:
        return f"{sign}${abs_value / 1_000_000:,.2f} million"
    if abs_value >= 1_000:
        return f"{sign}${abs_value / 1_000:,.2f} thousand"
    return f"{sign}${abs_value:,.2f}"


def pct_change(numerator, denominator):
    if denominator is None or pd.isna(denominator) or float(denominator) == 0.0:
        return pd.NA
    return 100.0 * float(numerator) / float(denominator)


# sector_ead_summary has one row per sector, so summing gives all-sector totals
baseline_total = float(sector_ead_summary['EAD_Baseline_USD'].sum())
deforestation_total = float(sector_ead_summary['EAD_Deforestation_USD'].sum())
reafforestation_total = float(sector_ead_summary['EAD_Reafforestation_USD'].sum())

change_deforestation_vs_baseline = deforestation_total - baseline_total
change_reafforestation_vs_baseline = reafforestation_total - baseline_total
benefit_reafforestation_vs_deforestation = deforestation_total - reafforestation_total

pct_deforestation_vs_baseline = pct_change(change_deforestation_vs_baseline, baseline_total)
pct_reafforestation_vs_baseline = pct_change(change_reafforestation_vs_baseline, baseline_total)
pct_benefit_vs_deforestation = pct_change(benefit_reafforestation_vs_deforestation, deforestation_total)

overall_totals = pd.DataFrame([
    {'Scenario': 'Baseline', 'Total_EAD_USD': baseline_total},
    {'Scenario': 'Deforestation', 'Total_EAD_USD': deforestation_total},
    {'Scenario': 'Reafforestation', 'Total_EAD_USD': reafforestation_total},
])
overall_totals['Total_EAD_USD_Readable'] = overall_totals['Total_EAD_USD'].apply(format_usd_readable)

overall_changes = pd.DataFrame([
    {
        'Metric': 'Deforestation change vs baseline',
        'Change_USD': change_deforestation_vs_baseline,
        'Percent': pct_deforestation_vs_baseline,
    },
    {
        'Metric': 'Reafforestation change vs baseline',
        'Change_USD': change_reafforestation_vs_baseline,
        'Percent': pct_reafforestation_vs_baseline,
    },
    {
        'Metric': 'Reafforestation benefit vs deforestation',
        'Change_USD': benefit_reafforestation_vs_deforestation,
        'Percent': pct_benefit_vs_deforestation,
    },
])
overall_changes['Change_USD_Readable'] = overall_changes['Change_USD'].apply(format_usd_readable)
overall_changes['Percent'] = overall_changes['Percent'].round(2)
overall_changes['Percent_Label'] = overall_changes['Percent'].apply(lambda x: 'NA' if pd.isna(x) else f"{x:.2f}%")

print('Total EAD across all sectors (USD):')
display(overall_totals)

print('Total changes across all sectors:')
display(overall_changes)

overall_totals_output_file = damage_estimates_path / 'landslide_ead_overall_totals_usd_combined_class.csv'
overall_changes_output_file = damage_estimates_path / 'landslide_ead_overall_changes_usd_with_pct_combined_class.csv'

overall_totals.to_csv(overall_totals_output_file, index=False)
overall_changes.to_csv(overall_changes_output_file, index=False)

print(f'Saved: {overall_totals_output_file}')
print(f'Saved: {overall_changes_output_file}')
